# model_03 - deneme 2 . model_a3'UN RECETESI

Kullanici karari, 17 Eylul 2026: *"a3'teki bir bulgu var ve onu
kullanacagiz ama model_a3 ile kiyaslama yapmayacagiz."*

Onceden kayit `belge/onkayit/model_03.md`.

### DUGME -- IKI ALAN, TEK RECETE

```
wd        0.1   ->  0.5      model_00'da 0.1
sabit_lr  False ->  True     cosine KAPALI, LR SABIT
```

`model_a3`in kazanci `model_a`ya gore tek dugmeydi (`wd`), ama
`model_a`nin arka planinda **zaten `sabit_lr=True` vardi**. `wd 0.5`i
cosine'in ustune koymak a3'un kosulunu kurmak DEGILDIR: cosine sonlara
dogru LR'yi onda birine indirir, yani agirlik cezasinin ezberi
eritmesini tam etkili olacagi yerde durdurur. Recete **ikisi birlikte**
alinir.

> **ATFETME YAPILAMAZ.** Iki alan birden degisiyor. CLAUDE.md:
> *"iki dugme birden -> ATFETME yapilamaz diye YAZILIR, kol kosulur."*

### NEDEN -- butun kollar `ent` ile dizilince

Dort kat butce `comp`i ucirdi ama `ent`e HICBIR SEY vermedi:

```
model_00 pencere      comp       ent
  12-20k             0.1303    0.0417
  32-40k             0.2060    0.0390
  52-60k             0.2703    0.0393
  72-80k             0.2760    0.0407     <- ent YERINDE SAYDI
```

Genel bir bilesim devresi kuruluyor olsaydi `ent` de yukselirdi: ayni
gorev, ayni iliskiler, tek fark oznenin egitimde zincir basi olup
olmamasi. Yukselmiyor. Yani butcenin satin aldigi sey **ozneye ozgu
istatistik**, devre degil. Kullanici, 17 Eylul: *"yavas degil limitli."*

```
kol                  ent      ne yapildi
model_01 wd 0.5    0.0040     2-hop YOK + agirlik cezasi
model_01           0.0117     2-hop YOK
model_b15          0.0290     dongu + Phi
model_02           0.0337     yumusak kopru dongusu
model_00           0.0407     klasik, DORT KAT butce
-----------------------------------------------------
model_b8           0.8650     KOPRU DENETIMI
model_a3           0.8743     wd 0.1 -> 0.5  (A ailesi)
```

Hepsi 0.03-0.04'te cakili; **sadece iki sey kirdi**. `b8` egitimde
koprunun etiketini ister. Digeri bu recete.

### !! BU RECETE TEK JETONLU ZINCIRDE CALISTI

Kullanici uyarisi, 17 Eylul. `model_a3`:

```
varliklar    TEK JETON        bizde UC YUVA
graf         veri_okul 1x     bizde veri_okul4, DORT KAT
ek / bicim   YOK / 1          bizde ek_kip=tr, bicim=3
```

Recetenin **zor dile tasinacaginin hicbir garantisi yok**. Bu kol tam
olarak onu siniyor.

Ikinci risk kendi kayitlarimizdan: CLAUDE.md kural 4'un gerekcesi
*"wd 0.5 + sabit LR rejiminde hicbir kol SAGLIK-1HOP ve SAGLIK-EZBER
kapilarini birlikte gecemedi"* diyor. Yani kol **on kosul kapilarinda
dusebilir** -- o zaman `comp` yorumlanmaz ve cevap "recete bu dilde
calismiyor" olur. Bu da bir sonuc.

### KIYAS

`model_00`, **esit butce**: 20.000 adim, pencere 12-20k.

```
model_00 12-20k   one 1.0000  seen 1.0000  comp 0.1303  ent 0.0417  ksy 0.0757
```

`model_a3` KIYAS TABANI DEGIL (kullanici karari) -- baska graf, baska
jetonlama, baska sinav. Sayilari yalnizca recetenin NEREDEN geldigini
anlatmak icin duruyor.

### KARAR KURALI - son pencere

```
comp        HUKUM                      (model_00 esit butce: 0.1303)
>= 0.50     OLGUNLUK KAPISI GECTI
0.30-0.50   BELIRGIN kazanc -- recete zor dile TASINDI
0.15-0.30   kismi
<= 0.15     recete bu dilde CALISMIYOR

on kosul    one >= 0.98 VE seen >= 0.95
            gecmezse comp YORUMLANMAZ
```

`ent` ve `ent_ksy` **raporlanir, hukum vermez** (kullanici karari,
17 Eylul: *"ent degil su anki hedef comp"*).

> **SAYISAL TAHMIN YAZILMIYOR** (kullanici karari, 15 Eylul).

### Beklenen kilit ciktisi

```
model_03/test_03.py    124 gecti, 0 BOZUK
  ok  wd 0.5 -- model_a3 recetesi
  ok  LR SABIT -- cosine KAPALI
  ok  RECETE TAM: wd 0.5 VE sabit LR -- yarisi alinmadi
  ok  model_b15'te wd 0.1 + cosine (DEGISMEDI)
  ok  EGITIM HAVUZU BIT DUZEYINDE ayni
  ok  OLCME IZI ayni  d6751004648c
```

---

**Sira:** 0 -> 1 -> 2 -> 3 -> 4 ile baslat. Kosu surerken **5 ve 6**.
Bitince **7** (`pencere_03`) ve **9** (`sor_03`).


In [ ]:
# 0 MODEL ADI VE YOLLAR  |  CPU  |  tekrar: GUVENLI
MODEL = "model_03"            # <-- DEGISTIRILECEK TEK SATIR

assert MODEL, ("MODEL bos. Bu SABLON -- kopyala, adini modelin adi yap "
               "(model_a.ipynb) ve bu satiri doldur.")
import time
DEPO  = "https://github.com/sekerahmet/sekerai.git"
KOD   = "/content/kod"
EV    = f"/content/drive/MyDrive/{MODEL}"     # TEPE KLASOR = MODELIN ADI
# LOG ADI BURADA URETILMEZ -- 4. hucre her BASLATMADA kendi damgasini
# basar. Sebep olculdu (15 Eylul): log adi burada uretilince, bu hucreyi
# yeniden calistirmadan ikinci bir kosu baslatmak ONCEKI kosunun logunu
# "w" ile ACIP SIFIRLIYOR. Fiilen oldu: 20.000'lik kosunun logu, 40.000'lik
# kosu baslayinca silindi. Damga BASLATAN hucrede uretilirse imkansiz.
print(MODEL, "->", EV)

In [ ]:
# 1 GPU VAR MI, BOS MU  |  GPU'yu SORAR, kullanmaz  |  tekrar: GUVENLI
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
_p = torch.cuda.get_device_properties(0)
_bos = torch.cuda.mem_get_info()[0] / 1e9
print(f"{_p.name}   toplam {_p.total_memory/1e9:.1f} GB   bos {_bos:.1f} GB")
assert _bos > 3.0, f"GPU'da sadece {_bos:.1f} GB bos -- baska bir sey mi kosuyor?"

In [ ]:
# 2 DRIVE  |  CPU  |  tekrar: GUVENLI (yalniz <model>/log/ acar)
from google.colab import drive
import os
drive.mount("/content/drive")
assert os.path.ismount("/content/drive"), \
    "drive.mount CALISMADI -- /content/drive gercek bir baglanti degil."
os.makedirs(f"{EV}/log", exist_ok=True)
with open(f"{EV}/.yazma_denemesi", "w") as f:
    f.write("ok")
os.remove(f"{EV}/.yazma_denemesi")
print("hazir ve YAZILABILIR:", EV)
print("  var olan tohum klasorleri:",
      sorted(d for d in os.listdir(EV) if d.startswith("t")) or "(yok)")

In [ ]:
# 3 KODU GITHUB'DAN CEK + KILIT TESTI  |  CPU  |  tekrar: KOSU YOKKEN (ilk isi rm -rf /content/kod)
import subprocess, os, glob, importlib, sys
subprocess.run(["rm", "-rf", KOD], check=True)
subprocess.run(["git", "clone", "--depth", "1", DEPO, KOD], check=True)
COMMIT = subprocess.run(["git", "-C", KOD, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("commit:", COMMIT, "|",
      subprocess.run(["git", "-C", KOD, "log", "-1", "--format=%s"],
                     capture_output=True, text=True).stdout.strip())

# Her modelin AILE klasoru var: deneme2/model_a/{model_a.py, pencere_a.py,
# model_a.ipynb}. Klasor adini isimden TURETMIYORUZ, dosyayi ARIYORUZ --
# model_a1 gibi varyasyonlar da ayni aile klasorunde durur.
# !! model_03 ARANMAZ: kendi klasoru BELLI. Kullanici karari,
# 16 Eylul -- "model_03 diger hicbir model ile ayni seyi kullanmamali".
# Paylasilan sablon `deneme2/*/<MODEL>.py` glob'u yapiyordu; model_03
# artik o aramaya girmiyor.
_aday = glob.glob(f"{KOD}/deneme2/model_03/{MODEL}.py")
assert len(_aday) == 1, f"{MODEL}.py tam bir kez bulunmali, bulunan: {_aday}"
AILE = os.path.dirname(_aday[0])
_pen = glob.glob(f"{AILE}/pencere_*.py")
assert len(_pen) == 1, f"ailede tam bir pencere_*.py olmali: {_pen}"
PENCERE = _pen[0]
print("aile:", AILE, "| olcum:", os.path.basename(PENCERE))

# --- IMPORT ONBELLEGINI TEMIZLE -- BU HUCRENIN EN SESSIZ TUZAGI ----------
# `rm -rf` + yeniden klon KODU tazeler ama `sys.modules` ESKI modul
# nesnesini tutar. Ayni cekirdekte MODEL degistirip bu hucreyi yeniden
# kosarsan, yeni kodu klonlamis ama ESKI modulu kullaniyor olursun.
# OLCULDU (15 Eylul): model_a4'ten model_a5'e gecerken
#   "AssertionError: Ayar'da boyle alan yok: {'ort_bas'}"
# cikti -- cunku model_a hala onceki klondan gelen, `ort_bas`i olmayan
# nesneydi. Daha sinsi hali: alan adlari tutarsa hata VERMEZ ve kosu
# ESKI KODLA baslar; kunyedeki commit ise YENIYI gosterir.
_atilan = [n for n, m in list(sys.modules.items())
           if getattr(m, "__file__", None) and str(m.__file__).startswith(KOD)]
for n in _atilan:
    del sys.modules[n]
importlib.invalidate_caches()
if _atilan:
    print("import onbellegi temizlendi:", sorted(_atilan))

# --- KILIT: test_03.py ---------------------------------------------------
# model_03 KENDI motoruna (taban_03.py) ve KENDI verisine (veri_03.py)
# sahip -- ikisi de model_a / veri_okul KOPYASI. `test_03.py` dort sey
# tutuyor: (0) BAGIMSIZLIK, model_03/*.py disariya import ETMIYOR;
# (1) MIMARI; (2) VERI, graf veri_okul4'unkiyle BIREBIR; (3) MOTOR,
# egitim havuzu ve olcme izi model_a ile AYNI.
# Duserse egitim BASLAMAMALI -- sayilar baska bir tabloda okunur.
#
# CIKTI YAKALANIR ve BASILIR: Colab alt surec stdout'unu hucreye
# aktarmiyor; "cikti yok" ile "test kosmadi" ayrimi sansa birakilmaz.
# AYRINTI=1 -> gecen kontroller de basilir (dugme tablosu gorunur olsun).
_t = [x for x in glob.glob(f"{AILE}/test_*.py")]
if _t:
    print()
    print("=" * 72)
    _r = subprocess.run([sys.executable, _t[0]], cwd=AILE,
                        capture_output=True, text=True,
                        env={**os.environ, "AYRINTI": "1",
                             "KOSU_KOK": EV.rsplit("/", 1)[0]})
    print(_r.stdout.rstrip() or "(cikti YOK -- test kosmamis olabilir!)")
    if _r.stderr.strip():
        print("stderr:", _r.stderr.rstrip()[-2000:])
    print("=" * 72)
    assert _r.returncode == 0, (
        f"{os.path.basename(_t[0])} DUSTU (cikis {_r.returncode}). Taban "
        f"degismis olabilir ve KAYITLI sonuclar ona dayaniyor. EGITIM BASLATMA.")
    print()

sys.path.insert(0, AILE)
# !! UST KLASOR EKLENMIYOR: veri modulu de (veri_03.py) AILE icinde.
# model_03 `deneme2/` kokundeki hicbir seyi gormez.
M_03 = importlib.import_module(MODEL)
M = M_03
# `M_03.M` MOTOR (taban_03). Paylasilan sablonda `M` TABANI
# gosteriyordu (model_a); burada `M` kolun KENDISI, motor ise
# `M_03.M`. Miras denetimi taban sinifi ORADAN alir.
MOTOR = M_03.M
assert M.AYAR.ad == MODEL, f"AYAR.ad {M.AYAR.ad!r} != dosya adi {MODEL!r}"
# Yuklenen modul GERCEKTEN yeni klondan mi geldi?
assert M.__file__.startswith(KOD), f"{MODEL} {KOD} disindan geldi: {M.__file__}"
# BU KOLUN SARTLARI. Hicbiri DEVRALINMIYOR: `ayar_03.py` her alani
# `Ayar()` varsayilaninin uzerine tek tek, gerekcesiyle yaziyor.
# --- MIMARI: bu kolun TANIMI --------------------------------------
assert M.AYAR.dongu == 1,              "DONGU YOK"
assert M.AYAR.l == 8,                  "8 AYRI katman"
assert M.AYAR.dar_alfa == 0.0,         "Phi DARBOGAZI YOK"
assert M.AYAR.dar_kapi is False,       "ogrenilen GECIT YOK"
assert M.AYAR.dff == 704,              "SwiGLU d_ff = 8/3*d"
assert M.AYAR.d // M.AYAR.nh == 64,    "head_dim 64"
# KUSUR (16 Eylul, defterin ILK kosusu): burada `M.Model` yaziyordu
# ve `M` = model_03 modulu; onda `Model` YOK -> AttributeError.
# Sablondan devralinmisti, defter hic kosulmadigi icin gorulmedi.
assert not issubclass(M_03.ModelSade, MOTOR.Model), \
    "ModelSade taban_03.Model den MIRAS ALMAMALI"
# ======================================================================
# BURADAN ASAGISI MIMARI DEGIL. Uc AYRI kategori, karistirilmasin:
#
#   A) DILIN KENDISI (korpus)  -- hiperparametre DEGIL. "ek_kip=tr"
#      demek "bu dil ekli bir dil" demek; "bicim=3" demek "ayni olgu
#      uc yuzey biciminde geciyor" demek. Modelin secimi degil,
#      METNIN ozelligi.
#   B) SINAV BOLMELERI          -- olcumun tanimi, modele ait DEGIL.
#   C) PROJEYE OZGU VERI EKI    -- BEST PRACTICE DEGIL. Tek kalem:
#      identity bridge (ident_frac / ident_kip). model_b13'te
#      arXiv 2509.24653'ten alindi. Burada DURUYOR cunku egitim havuzu
#      model_b15 ile BIT DUZEYINDE ayni olmali; olmazsa kol "mimari mi
#      veri mi" sorusunu AYIRAMAZ. Kaldirilirsa kol AYRI bir soru sorar
#      (onkayit model_03.md 2).
#
# Optimizasyon (wd/cosine/isinma/lr/betas) YUKARIDA degil ASAGIDA ve
# hepsi STANDART TARIF -- proje kisiti DEGIL.
# ======================================================================
# --- A) DILIN KENDISI -------------------------------------------------
# --- BU KOLUN TEK DUGMESI ------------------------------------------
assert M.AYAR.wd == 0.5,               "model_a3 RECETESI -- kolun TANIMI"
assert M.AYAR.sabit_lr is True,        "LR SABIT -- recetenin ikinci yarisi"
assert not hasattr(M.AYAR, "dusun_gecis"), "model_03te DONGU YOK"
assert M.AYAR.veri_ad == "veri_03",    "model_03 KENDI veri modulu"
import veri_03 as _V03
assert _V03.graf_izi(_V03.kur(0)) == _V03.IZ, "graf KAYMIS"
print(f"veri_03  graf izi {_V03.IZ}  (icerik veri_okul4, 1060 varlik)")
assert M.AYAR.jeton_ad == "tam",       "varlik = JETON DIZISI (3 yuva)"
assert M.AYAR.ek_kip == "tr",          "dil EKLI: '  <NIN>  <SI>  <DIR>"
assert M.AYAR.bicim == 3,              "ayni olgu UC yuzey biciminde"
assert M.AYAR.t_len == 17,             f"t_len DEGISMEDI -- SINAV AYNI: {M.AYAR.t_len}"
assert M.AYAR.belge_pay == 0.0,        "BELGE satiri YOK (ek_kip ile kurulmadi)"

# --- B) SINAV BOLMELERI -- olcumun tanimi -----------------------------
assert M.AYAR.ood_pay == 0.05,         f"ood bolmesi: {M.AYAR.ood_pay}"
assert M.AYAR.kopru_kayip == 0.0,      "YARDIMCI KAYIP YOK (bu MIMARI karari)"

# --- C) PROJEYE OZGU VERI EKI -- BEST PRACTICE DEGIL ------------------
# Kullanici, 16 Eylul: "biz her seyi sifirdan yaptik, ben kisit
# vermedim, best practice dedim." Dogru -- ve bu IKI SATIR o tarifin
# parcasi DEGIL. Egitim havuzu b15 ile ayni kalsin diye duruyor.
assert M.AYAR.ident_frac == 0.2,       "identity bridge -- PROJE EKI, b15 ile AYNI"
assert M.AYAR.ident_kip == "q1",       "identity bridge -- PROJE EKI, b15 ile AYNI"

# --- OPTIMIZASYON: STANDART TARIFIN KENDISI ---------------------------
# Bunlar proje kisiti DEGIL. Dordu de ayni sayilari kullaniyor:
#   nanoGPT   GPT-3   Llama   Pythia
assert M.AYAR.lr == 1e-3,              "Pythia-70m ile ayni mertebe"
assert M.AYAR.isinma == 2000,          "nanoGPT warmup_iters=2000, Llama 2000"
assert M.AYAR.betas == (0.9, 0.95),    "nanoGPT/GPT-3/Llama/Pythia -- 0.999 DEGIL"
assert M.AYAR.tam_kayip is True,       "butun pozisyonlarda next-token: standart LM"
# PROJEYE OZGU OPTIMIZASYON NUMARALARI -- HEPSI KAPALI
assert M.AYAR.ort_bas == 0,            "LOOKAHEAD KAPALI -- hicbir tarifte YOK"
assert not (M.AYAR.dar_sert or M.AYAR.dar_sdpa), "darbogaz zaten YOK"
print("mimari: ModelSade  8 katman  RoPE  SwiGLU  bagli gomme  bias YOK")
SOR = os.path.join(AILE, "sor_03.py")  # 9. hucre kullanir
for _g in ("AYAR", "egit", "fark_bas"):
    assert hasattr(M, _g), f"{MODEL}'de {_g} YOK -- kos_03.py duser"
print("ayar:", M.AYAR)

In [ ]:
# 4 BASLAT  |  GPU (alt surec)  |  tekrar: HAYIR -- yeni kosu baslatir
# --- GPU KAPISI (CLAUDE.md kural 2) -------------------------------------
# GPU kullanacak her hucre, ONCE kendi icinde GPU'yu sorar. Ayri bir
# "GPU var mi" hucresi HUCRE SIRASINA bagli bir kuraldir; icerideki
# kontrol degildir. Olculdu (15 Eylul): Colab cekirdegi kosu sirasinda
# oldu, GPU ve Drive durumu gitti; ayri kontrol hucresi vardi ama
# calistirilmamisti.
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, f"GPU'da sadece {_bos:.1f} GB bos -- baska bir sey mi kosuyor?"
print(f"GPU kapisi GECTI: {torch.cuda.get_device_name(0)}  bos {_bos:.1f} GB")
# ------------------------------------------------------------------------

TOHUMLAR = [0]             # ONCE TEK TOHUM.
# !! BU KOLUN OLCUTU `comp` (ent DEGIL) -- onkayit model_03.md.
# Sonuc OLUMLU cikarsa (comp yukseldiyse) tek tohum YETER.
# OLUMSUZ cikarsa bu satiri [1, 2] yapip tekrar kos -- t0 klasoru
# DOKUNULMAZ, yeni kosular t1/ ve t2/'ye yazar. Kod degismez.
# Neden onemli: grokking tohuma bagli. Tek tohumda comp yukselmezse
# "konfigurasyon yanlis" ile "bu baslangic sanssiz" AYRILAMAZ.
# VERI tohumu AYRI (ayar.veri_tohum=0) -- butun tohumlar AYNI veriyi gorur.

# !! ADIM buyutup SURDUR=True yapmak UZATMADIR, sifirdan kosu DEGIL.
#    Ama cosine ufku `ayar.adim`dan turedigi icin LR GERI FIRLAR:
#    model_b14'te 20.000 ufkunda lr/10 iken 60.000 ufkunda 7,6 KAT.
#    Uzatilmis kosu, temiz bir uzun kosu DEGILDIR.
ADIM   = None    # None = ayar_03.py'deki ILK SINIR (20.000).
#                  Uzatmak KULLANICI karari (CLAUDE.md kural 1).
SURDUR = False   # TAZE kosu -- surdurme DEGIL.
USTUNE = False   # t0 BOS (yeni kol). True = dolu klasoru
#                  t<N>_eski_<zaman>/'a TASI (silmez)

if "p" in globals() and p.poll() is None:
    raise SystemExit(f"ZATEN KOSUYOR (PID {p.pid}). Once 'Durdurmak' hucresi.")

# KENDI kosucusu: model_03/kos_03.py. `--model` YOK -- bu betik
# yalnizca model_03'i baslatir (paylasilan kos.py aile klasoru ARIYORDU).
_arg = [sys.executable, "-u", f"{KOD}/deneme2/model_03/kos_03.py",
        "--ev", EV, "--commit", COMMIT,
        "--tohum", *[str(t) for t in TOHUMLAR]]
if ADIM:
    _arg += ["--adim", str(ADIM)]
if SURDUR:
    _arg += ["--surdur"]
if USTUNE:
    _arg += ["--ustune"]
# LOG ADI HER BASLATMADA YENI: bu hucre iki kez calisirsa iki AYRI log
# olur, oncekinin uzerine YAZILMAZ.
LOG = f"{EV}/log/kos_{time.strftime('%Y%m%d_%H%M%S')}.txt"
p = subprocess.Popen(_arg, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)
print("PID", p.pid, " tohum", TOHUMLAR, " -> log:", LOG)
print("Dolu bir tohum klasoru varsa kosu REDDEDILIR -- hicbir sey ezilmez.")
print("Ayni tohumu bilerek tekrar kosmak icin --ustune; o da SILMEZ,")
print("eskisini t<N>_eski_<zaman>/ diye yan klasore TASIR.")

In [ ]:
# 5 ILERLEME (ham log)  |  CPU  |  tekrar: GUVENLI (log geriden gelebilir)
import glob, subprocess
# LOG degiskenine DEGIL, klasordeki EN YENI log'a bak.
_l = sorted(glob.glob(f"{EV}/log/kos_*.txt"))
assert _l, f"log yok: {EV}/log/"

# `p` CEKIRDEK YENIDEN BASLAYINCA KAYBOLUR -- ve tam o an bu hucreye
# ihtiyac duyulur. Olculdu (15 Eylul): Colab cekirdegi oldu, bu hucre
# `NameError: name 'p' is not defined` verdi, koşunun yasayip yasamadigi
# ogrenilemedi. Artik `p` yoksa SUREC TABLOSUNA bakiyor.
if "p" in globals():
    _d = p.poll()
    print("KOSUYOR" if _d is None else f"BITTI/OLDU (cikis kodu {_d})",
          "| PID", p.pid)
else:
    _ps = subprocess.run(
        ["bash", "-lc", "ps -eo pid,etime,cmd | grep kos_03.py | grep -v grep"],
        capture_output=True, text=True).stdout.strip()
    print("!! `p` YOK -- cekirdek yeniden baslamis.")
    if _ps:
        print("   ama SUREC YASIYOR:\n   " + _ps)
    else:
        print("   ve kos_03.py sureci de YOK -> kosu OLDU.")
        print("   Drive'i yeniden bagla (2. hucre), 3'u kos, sonra 4. hucrede")
        print("   SURDUR = True ile KALDIGI YERDEN devam ettir.")
print("log:", _l[-1].split("/")[-1], f"({len(_l)} log dosyasi var)")
print("-" * 78)
print(open(_l[-1]).read()[-4000:])

In [ ]:
# 6 RAPOR (canli durum)  |  CPU  |  tekrar: GUVENLI
import json, glob, os, statistics

# KIYAS TABANI: model_00. AYNI MIMARI, AYNI VERI, AYNI SINAV
# (iz d6751004648c), AYNI EGITIM HAVUZU, AYNI dizi uzunlugu.
# FARK: OPTIMIZASYON -- wd 0.1 -> 0.5 ve cosine -> SABIT LR.
# Iki alan birden, ama TEK RECETE (model_a3'un kosulu). ATFETME YAPILAMAZ.
# !! ESIT BUTCE. model_03 20.000'de duruyor, o yuzden taban model_00'in
# AYNI butcedeki penceresi. 72-80k (comp 0.2760) DORT KAT butce -- ayri
# satirda, hukumde KULLANILMAZ.
M00 = dict(one=1.0000, seen=1.0000, comp=0.1303, ood=0.0333,
           ent=0.0417, ent_yok=0.0429)      # model_00 PENCERE 12-20k
M00_UZUN_COMP = 0.2760                      # model_00 72-80k -- BILGI icin
B15 = dict(one=0.9997, seen=0.9997, comp=0.0550, ood=0.0143,
           ent=0.0290, ent_yok=0.0409)      # model_b15 PENCERE 12-20k
# !! model_b8 BASKA BIR SINAVDA kostu: veri_okul2, bicim=1, ek_kip YOK.
# Yani bu sutun model_00/model_03 ile SATIR SATIR kiyaslanamaz --
# yalnizca "gorev cozulebilir mi" sorusuna TAVAN olarak bakilir.
# (Kusur 17 Eylul: sutun basligi 'b8 TAVAN' idi ve ayni sinavmis gibi
#  duruyordu.)
B8 = dict(one=0.9510, seen=0.9913, comp=0.9747, ood=0.0177,
          ent=0.8650, ent_yok=0.8710)       # kopru EGITIMDE zorlandi
M00_KSY = 0.0757                            # model_00 12-20k kisayol
M00_MS = 74.2                               # model_00 OLCULDU (t_len 17)

_TUM = (("adim", "adim"), ("kayip", "kayip"), ("one", "one"),
        ("seen", "seen"), ("comp", "comp"), ("ood", "ood"), ("ent", "ent"),
        ("ent_kati", "ent_kati"), ("ent_yok", "ent_yok"),
        ("kayip_ana", "kayip_ana"),
        ("ent_kisayol", "ent_ksy"), ("ent_yok_kisayol", "yok_ksy"))

for kl in sorted(k for k in glob.glob(f"{EV}/t*") if os.path.isdir(k)):
    eg = glob.glob(f"{kl}/egri_*.json")
    if not eg:
        print(f"{os.path.basename(kl)}: egri YOK"); continue
    ky = glob.glob(f"{kl}/kosu_t*.json")
    k = json.load(open(ky[0])) if ky else {}
    e = json.load(open(eg[0]))
    _var = set().union(*(set(r) for r in e))
    SUT = tuple(x for x in _TUM if x[0] in _var)
    _atlanan = [c for c in _var
                if c.startswith(("one", "seen", "comp", "ood", "ent"))
                and c not in {x[0] for x in SUT}]
    assert not _atlanan, f"OLCULUYOR AMA BASILMIYOR: {_atlanan}"
    _iz = k.get("olcme_izi", "?")
    assert _iz == "d6751004648c", (
        f"olcme izi {_iz} != d6751004648c -- model_00 KIYASI GECERSIZ")
    print("=" * 84)
    print(f"{os.path.basename(kl)}   {k.get('ad','?')}   durum {k.get('durum','?')}"
          f"   commit {k.get('commit','?')}   {k.get('gpu','')}")
    v = k.get("veri", {})
    if v:
        print(f"   olgu {v['olgu']}  egitim2 {v['egitim2']}  ENT {v['ent']}  "
              f"phi {v['phi']} (wang {v['wang_phi']})  "
              f"parametre {k.get('parametre',0):,}  iz {k.get('olcme_izi','?')}")
    print("   " + "".join(f"{b:>10}" for _, b in SUT) + f"{'dk':>6}")
    for r in e:
        hcr = []
        for c, _ in SUT:
            x = r.get(c)
            hcr.append(f"{x:>10d}" if c == "adim" else
                       (f"{'---':>10}" if x is None else f"{x:>10.4f}"))
        print("   " + "".join(hcr) + f"{r['sn']/60:>6.0f}")

    s = e[-1]
    print("   " + "-" * 81)
    # !! BU BLOK EGRI OKUMASI -- son olcum noktasinin TEK anlik goruntusu.
    # HUKUM PENCEREYLE verilir (7. hucre, pencere_03). Ikisi AYRI sey ve
    # AYRISABILIR: model_03'te egri one 0.9153 / seen 0.9210 ile kapilarda
    # KALDI, pencere 0.9857 / 0.9960 ile GECTI. Agirlik ortalamasi
    # gurultuyu siliyor. CLAUDE.md kural 3: okuma yontemi kolun sorusuna
    # gore secilir ve onkayit model_03.md 4 PENCEREYI yaziyor.
    # (Kusur 17 Eylul: bu blok "KALDI" basiyordu ve hukummus gibi
    #  okunabilirdi.)
    print("   [EGRI okumasi -- HUKUM DEGIL. Hukum: 7. hucre, pencere_03]")
    print(f"   {'GECTI ' if s.get('one',0) >= 0.98 else '!! KALDI'}"
          f"  {'SAGLIK-1HOP':<14} one >= 0.98      <- ON KOSUL")
    print(f"   {'GECTI ' if s.get('seen',0) >= 0.95 else '!! KALDI'}"
          f"  {'SAGLIK-EZBER':<14} seen >= 0.95     <- ON KOSUL")
    print(f"   {'GECTI ' if s.get('comp',0) >= 0.50 else '!! KALDI'}"
          f"  {'OLGUNLUK':<14} comp >= 0.50")
    print(f"   {'GECTI ' if abs(s.get('ent_yok_kisayol',1)) < 1e-9 else '!! KALDI'}"
          f"  {'BIRIM TESTI':<14} ent_yok_kisayol == 0")

    print("   " + "-" * 81)
    d = [(b["sn"] - a["sn"]) / (b["adim"] - a["adim"]) * 1000
         for a, b in zip(e, e[1:])
         if b["sn"] > a["sn"] and b["adim"] > a["adim"]]
    if d:
        ms = statistics.median(d)
        print(f"   HIZ ortanca {ms:.1f} ms/adim   (model_00 {M00_MS})"
              f"   {(ms/M00_MS-1)*100:+.1f}%   [AYNI mimari -- fark yalniz"
              f" OPTIMIZASYON, maliyet DEGISMEMELI]")
    print(f"   {'olcu':<9}{'b15 pen':>10}{'model_00':>11}{'model_03':>10}"
          f"{'b8 !BSK':>10}")
    for c in ("one", "seen", "comp", "ood", "ent", "ent_yok"):
        if c in s:
            print(f"   {c:<9}{B15[c]:>10.4f}{M00[c]:>11.4f}{s[c]:>10.4f}"
                  f"{B8[c]:>10.4f}")
    print("   b8 !BSK = BASKA SINAV (veri_okul2, bicim=1, ek_kip YOK).")
    print("   Kiyas DEGIL, yalnizca gorevin cozulebilirligini gosterir.")
    print("   model_00 sutunu ESIT BUTCE penceresi (12-20k). Dort kat")
    print(f"   butcede comp {M00_UZUN_COMP:.4f} idi -- hukumde KULLANILMAZ.")
    print("   model_03 ile model_00 farki: wd 0.5 + sabit LR.")
    if "comp" in s:
        c_ = s["comp"]
        print("   BIRINCIL (comp -- onkayit model_03.md 6): "
              + ("OLGUNLUK KAPISI GECTI" if c_ >= 0.50 else
                 "0.30-0.50 BELIRGIN kazanc -- recete ZOR DILE TASINDI"
                 if c_ >= 0.30 else
                 "0.15-0.30 kismi" if c_ > 0.15 else
                 "<=0.15 recete bu dilde CALISMIYOR"))
        print("      ON KOSUL: one VE seen. Gecmezse comp YORUMLANMAZ.")
        print(f"      model_00 comp {M00['comp']:.4f} -> model_03 {c_:.4f}"
              f"   fark {c_-M00['comp']:+.4f}")
    if "ent_kisayol" in s:
        ksy = s["ent_kisayol"]
        print(f"   KISAYOL ent_ksy {ksy:.4f}  (model_00 {M00_KSY:.4f})"
              f"   fark {ksy-M00_KSY:+.4f}")
        print("      model_a3'te bu olcu 0.2337 -> 0.0027 dusmustu (86 kat).")
        print("      RAPORLANIR, HUKUM VERMEZ -- hedef comp (kullanici, 17 Eylul).")
    if "ent" in s:
        print(f"   ent {s['ent']:.4f} (model_00 {M00['ent']:.4f})"
              f"  -- RAPORLANIR, HUKUM VERMEZ")
    if "ood" in s:
        print(f"   ood {s['ood']:.4f} -- RAPORLANIR, HUKUM VERMEZ. 210 ornek.")


In [ ]:
# 7 pencere_03 -- BIRINCIL OKUMA  |  GPU  |  tekrar: GUVENLI, ama ANCAK KOSU BITINCE
# --- GPU KAPISI (CLAUDE.md kural 2) -------------------------------------
# GPU kullanacak her hucre, ONCE kendi icinde GPU'yu sorar. Ayri bir
# "GPU var mi" hucresi HUCRE SIRASINA bagli bir kuraldir; icerideki
# kontrol degildir. Olculdu (15 Eylul): Colab cekirdegi kosu sirasinda
# oldu, GPU ve Drive durumu gitti; ayri kontrol hucresi vardi ama
# calistirilmamisti.
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, f"GPU'da sadece {_bos:.1f} GB bos -- baska bir sey mi kosuyor?"
print(f"GPU kapisi GECTI: {torch.cuda.get_device_name(0)}  bos {_bos:.1f} GB")
# ------------------------------------------------------------------------

# BIRINCIL OKUMA. GPU kullanir (yukaridaki nota bak).
# PENCERE hucre 3'te ailenin icinden bulundu -- yolu elle yazmiyoruz.
TOHUM = TOHUMLAR[0]
!python {PENCERE} {EV}/t{TOHUM} --genislik 5

In [ ]:
# 8 DURDUR  |  CPU  |  tekrar: KOSUYU OLDURUR -- bastaki # bilerek duruyor
# Anlik goruntuler Drive'da kalir; surdurme paketi her olcum
# noktasinda yazilir, 4. hucrede SURDUR=True ile devam edilir.
# p.kill()

In [ ]:
# 9 sor_03 -- KENDI SORUNU SOR  |  GPU  |  tekrar: GUVENLI (OLCU DEGIL)
# --- GPU KAPISI (CLAUDE.md kural 2) -------------------------------------
import torch
assert torch.cuda.is_available(), "GPU YOK"
assert torch.cuda.mem_get_info()[0] / 1e9 > 2.0, "GPU dolu"
# ------------------------------------------------------------------------
# KENDI SORUNU SOR -- TURKCE. Listeye istedigin kadar satir ekle.
# Turkce harf SART DEGIL: "kardesi" de olur "kardesi" de (cozumleyici
# 1161 yazimi taniyor, hicbir ikisi cakismiyor).
# Ciplak yazim da calisir: "Ahmet Yilmaz anne kardes".
# !! BU BIR OLCU DEGIL -- elle sorulan sorular SECILMIS sorulardir.
SORULAR = [
    "Ahmet Yilmaz'in annesi",                    # 1-hop
    "Ahmet Yilmaz'in annesinin kardesi",         # 2-hop
    "Ahmet Yilmaz'in kardesinin okulu",          # 2-hop, cevap OKUL
    "Adana Lisesi'nin muduru",                   # 1-hop, OKUL -> KISI
    "Matematik'in hocasi",                       # 1-hop, DERS -> KISI
    "Adana'nin komsusunun valisi",               # 2-hop, SEHIR zinciri
]
_q = " ".join(f'--soru "{s}"' for s in SORULAR)
!python {SOR} {EV}/t{TOHUMLAR[0]} --genislik 5 {_q}

In [ ]:
# 10 asama1_03 --sonda -- KOPRU SONDASI  |  GPU  |  tekrar: GUVENLI
# --- GPU KAPISI (CLAUDE.md kural 2) -------------------------------------
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, f"GPU'da sadece {_bos:.1f} GB bos"
print(f"GPU kapisi GECTI: {torch.cuda.get_device_name(0)}  bos {_bos:.1f} GB")
# ------------------------------------------------------------------------
# ASAMA-1 TESHISI -- model_03'da kopru hidden state'e girdi mi?
# !! BU KOLUN EK OKUMASI BURADA: Physics 3.1'in iddiasi
# 'augmentation olmadan bilgi EZBERLENIR ama DOGRUSAL KODLANMAZ'.
# Sonda slot 0 'bilgi VAR' derse comp acilmasa bile bu bir bulgu.
# arXiv 2505.17923 AYNI probe'u AYNI pozisyonda yapmis ve kopruyu
# BULMUS: 'the hidden representation of the last input token
# encodes information about all necessary bridge entities'.
# model_b13'te ayni pozisyonda 0.0275 cikmisti.
# MERDIVEN: model_b6 -> b9 -> model_b10 (TABAN) -> b13 -> b8.
#   model_b9 comp 0.0250 | model_b6 0.0442 | model_b8 TAVAN 0.9997
# --sonda : kopru DOGRUSAL okunabiliyor mu. Taban max(en_sik, KOPYA) --
#           kopru cogu zaman soru varligiyla ayni aileden, soyad GIRDIDE
#           duruyor (comp'ta kopya 0.4450). Bu duzeltilmeden slot1
#           yanlislikla "BILGI VAR" cikiyordu.

import os
ASAMA1 = os.path.join(AILE, "asama1_03.py")
assert os.path.exists(ASAMA1), ASAMA1
!python {ASAMA1} {EV}/t{TOHUMLAR[0]} --genislik 5 --sonda --birim --bolme comp,ent,ood,seen

In [ ]:
# 11 tani_03 -- ARIZA SEKLI  |  GPU  |  tekrar: GUVENLI
# --- GPU KAPISI (CLAUDE.md kural 2) -------------------------------------
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, f"GPU'da sadece {_bos:.1f} GB bos"
print(f"GPU kapisi GECTI: {torch.cuda.get_device_name(0)}  bos {_bos:.1f} GB")
# ------------------------------------------------------------------------
# AYRISTIRMA: model YANLIS cevap verirken NE diyor?
#   KISAYOL (r2'yi dogrudan soru varligina uygulamis)
#   KOPRU   (ara varligi yazmis, ikinci hop'u yapmamis)
#   VARLIK_DEGIL / ILGISIZ / ...
# Ayrica: 1.hop tek basina, 2.hop tek basina, IKISI BIRDEN -> KAYIP.
# Bu bir HUKUM olcusu DEGIL, arizanin SEKLINI gosterir.
import os
TANI = os.path.join(AILE, "tani_03.py")
assert os.path.exists(TANI), TANI
!python {TANI} {EV}/t{TOHUMLAR[0]} --genislik 5

In [ ]:
# 12 tani_b -- model_b15 TABAN KIYASI  |  GPU  |  tekrar: GUVENLI
# --- GPU KAPISI (CLAUDE.md kural 2) -------------------------------------
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, f"GPU'da sadece {_bos:.1f} GB bos"
print(f"GPU kapisi GECTI: {torch.cuda.get_device_name(0)}  bos {_bos:.1f} GB")
# ------------------------------------------------------------------------
# TABAN KIYASI: model_03'in ariza SEKLINI model_b15 ile kiyasla.
# !! AYNI SINAV KUMESI (iz d6751004648c) -- sayilar DOGRUDAN
#    kiyaslanabilir. Degisen YALNIZ mimari.
# model_b15 bir ModelB oldugu icin onu `tani_b.py` ile okuyoruz.
# Bu kolun ariza SEKLI model_b15'ten FARKLI mi, yoksa ayni mi?
# Ayniysa "mimari hicbir seyi degistirmedi" DAHA GUCLU soylenir.
# EGITIM YOK, yalniz okuma -- model_b15/t0 Drive'da duruyor.
import os
TANI_B = "/content/kod/deneme2/model_b/tani_b.py"
EV_B6 = os.path.dirname(EV) + "/model_b15"
assert os.path.isdir(f"{EV_B6}/t0"), f"model_b15/t0 YOK: {EV_B6}"
!python {TANI_B} {EV_B6}/t0 --genislik 5